# Phase 2 — Extraction intelligente ESG depuis les PDFs collectés

## Positionnement dans le projet

Ce notebook est la suite directe de `ESG_quant_project.ipynb`.

La phase 1 produit un corpus documentaire ESG : PDFs téléchargés, logs, métadonnées, score documentaire, statut de sélection et hash SHA-256. La phase 2 transforme ce corpus en données structurées, auditables et exploitables pour le scoring ESG, l'analyse de risque climat et les signaux quantitatifs.

La démarche est conçue pour un Data Case SCOR : elle privilégie la traçabilité, la prudence méthodologique, la validation métier et la conservation de la preuve documentaire.

## Chaîne cible

```text
Phase 1 : discovery → validation PDF → scoring documentaire → sélection → ingestion
Phase 2 : parsing PDF → texte/tables/images → routing ESG → extraction métriques → validation → panel entreprise-année-indicateur
Phase 3 : scoring ESG → signaux de risque / investissement durable
```

Chaque métrique extraite conserve : document source, entreprise, exercice, page, bloc, extrait, méthode d'extraction, score de confiance et statut de validation.


## 1. Imports, dépendances et politique d'exécution

Cette cellule charge les dépendances nécessaires sans forcer d'installation réseau au moment de l'exécution.

Pourquoi ce choix ? La phase 1 installait automatiquement les packages dans le notebook, ce qui est pratique en exploration. Pour une phase d'extraction plus sérieuse et présentable en entretien, les dépendances sont désormais listées dans `requirements-phase2.txt`, ce qui rend l'environnement plus reproductible.

Les dépendances PDF/OCR sont traitées proprement : si une librairie optionnelle est absente, le notebook indique ce qui manque et les fonctions concernées renverront un message explicite plutôt que de bloquer silencieusement.


In [ ]:
# ============================================================
# 1. IMPORTS, DÉPENDANCES ET POLITIQUE D'EXÉCUTION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Tuple

import hashlib
import importlib
import json
import math
import re
import shutil
import traceback
import uuid


def optional_import(import_name: str, package_hint: Optional[str] = None):
    """Importe un module optionnel et retourne None s'il est absent."""
    try:
        return importlib.import_module(import_name)
    except ImportError:
        hint = package_hint or import_name
        print(f"[WARN] Module absent : {import_name}. Installer via `pip install {hint}` si nécessaire.")
        return None


pd = optional_import("pandas")
fitz = optional_import("fitz", "pymupdf")
pdfplumber = optional_import("pdfplumber")
pyarrow = optional_import("pyarrow")
PIL_Image = optional_import("PIL.Image", "Pillow")
pytesseract = optional_import("pytesseract")

REQUIRED_FOR_FULL_RUN = {
    "pandas": pd,
    "pymupdf/fitz": fitz,
    "pdfplumber": pdfplumber,
}

missing_required = [name for name, module in REQUIRED_FOR_FULL_RUN.items() if module is None]
if missing_required:
    print("[INFO] Dépendances manquantes pour l'exécution complète :", missing_required)
    print("[INFO] Installer les dépendances avec : pip install -r requirements-phase2.txt")
else:
    print("[OK] Dépendances principales disponibles")

OCR_ENGINE_AVAILABLE = pytesseract is not None and shutil.which("tesseract") is not None
print(f"OCR système disponible : {OCR_ENGINE_AVAILABLE}")


## 2. Configuration héritée de la phase 1 et nouveaux répertoires

Cette cellule reprend l'arborescence de la phase 1 :
- `esg_data/raw` pour les PDFs ;
- `esg_data/metadata/documents_metadata.csv` pour les métadonnées d'ingestion ;
- `esg_data/metadata/selected_documents.csv` pour les décisions de sélection.

La phase 2 ajoute des sorties dédiées : pages parsées, blocs, tables, images, candidats métriques, métriques validées, file de revue et panel final.


In [ ]:
# ============================================================
# 2. CONFIGURATION DES CHEMINS
# ============================================================

BASE_DIR = Path("esg_data")
RAW_DIR = BASE_DIR / "raw"
LOG_DIR = BASE_DIR / "logs"
META_DIR = BASE_DIR / "metadata"

PHASE2_DIR = BASE_DIR / "phase2_extraction"
PARSED_DIR = PHASE2_DIR / "parsed"
EXTRACTED_DIR = PHASE2_DIR / "extracted"
REVIEW_DIR = PHASE2_DIR / "review"
QUALITY_DIR = PHASE2_DIR / "quality"
IMAGE_DIR = PARSED_DIR / "images"

for directory in [LOG_DIR, META_DIR, PHASE2_DIR, PARSED_DIR, EXTRACTED_DIR, REVIEW_DIR, QUALITY_DIR, IMAGE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Entrées produites par la phase 1
DOCUMENT_METADATA_PATH = META_DIR / "documents_metadata.csv"
SELECTED_DOCUMENTS_PATH = META_DIR / "selected_documents.csv"

# Registre phase 2 versionné dans le repository
METRIC_REGISTRY_PATH = Path("data/esg_metric_registry_catalog.csv")

# Sorties phase 2
PARSED_PAGES_PATH = PARSED_DIR / "parsed_pages.parquet"
DOCUMENT_BLOCKS_PATH = PARSED_DIR / "document_blocks.parquet"
EXTRACTED_TABLES_PATH = PARSED_DIR / "extracted_tables.parquet"
EXTRACTED_IMAGES_PATH = PARSED_DIR / "extracted_images.parquet"
METRIC_CANDIDATES_PATH = EXTRACTED_DIR / "metric_candidates.parquet"
VALIDATED_METRICS_PATH = EXTRACTED_DIR / "validated_metrics.parquet"
COMPANY_YEAR_PANEL_PATH = EXTRACTED_DIR / "company_year_esg_panel.parquet"
EXTRACTION_LOG_PATH = LOG_DIR / "phase2_extraction_log.csv"
REVIEW_QUEUE_PATH = REVIEW_DIR / "metric_review_queue.csv"
QUALITY_REPORT_PATH = QUALITY_DIR / "phase2_quality_report.csv"

# Paramètres d'exécution
MAX_PAGES_PER_DOCUMENT = None       # Mettre 20 ou 50 pour un smoke test rapide
ENABLE_OCR = False                  # Passer à True si Tesseract est installé
MIN_TEXT_CHARS_FOR_PAGE = 40
CONTEXT_WINDOW_CHARS = 450


## 3. Schémas explicites des sorties

Comme dans la phase 1, les schémas sont déclarés explicitement. Cela permet de garantir une structure stable entre les runs et de rendre les sorties directement exploitables par la suite du projet.

La conception suit une logique de preuve : une métrique n'est jamais seulement une valeur, c'est une valeur liée à une page, un bloc, une méthode d'extraction et un niveau de confiance.


In [ ]:
# ============================================================
# 3. SCHÉMAS DES SORTIES PHASE 2
# ============================================================

EXTRACTION_LOG_COLUMNS = [
    "run_id", "stage", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "status", "error_type", "error_message", "created_at"
]

PARSED_PAGE_COLUMNS = [
    "run_id", "document_id", "company", "ticker", "isin", "jurisdiction", "fiscal_year",
    "doc_subtype", "source_pdf_path", "sha256", "page_number", "text", "text_length",
    "has_tables", "has_images", "language", "section_labels", "section_scores",
    "extraction_quality_score", "created_at"
]

DOCUMENT_BLOCK_COLUMNS = [
    "run_id", "block_id", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "page_number", "block_type", "block_order", "content", "bbox",
    "section_labels", "confidence", "created_at"
]

EXTRACTED_TABLE_COLUMNS = [
    "run_id", "table_id", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "page_number", "table_order", "n_rows", "n_cols", "table_json",
    "extraction_method", "created_at"
]

EXTRACTED_IMAGE_COLUMNS = [
    "run_id", "image_id", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "page_number", "image_order", "image_path", "width", "height",
    "ocr_text", "ocr_status", "created_at"
]

METRIC_CANDIDATE_COLUMNS = [
    "run_id", "metric_id", "document_id", "company", "ticker", "isin", "jurisdiction",
    "fiscal_year", "doc_subtype", "metric_name", "metric_category", "raw_value", "raw_unit",
    "normalized_value", "normalized_unit", "source_page", "source_block_id",
    "source_text_excerpt", "extraction_method", "confidence_score", "validation_status",
    "quality_flags", "created_at"
]

COMPANY_YEAR_PANEL_COLUMNS = [
    "run_id", "company", "ticker", "isin", "jurisdiction", "fiscal_year", "metric_name",
    "metric_category", "metric_value", "metric_unit", "best_confidence_score", "validation_status",
    "source_document_id", "source_doc_subtype", "source_page", "source_text_excerpt",
    "quality_flags", "created_at"
]

QUALITY_REPORT_COLUMNS = [
    "run_id", "level", "key", "n_documents", "n_pages", "n_tables", "n_images",
    "n_metric_candidates", "n_validated_metrics", "n_review_required", "coverage_ratio",
    "created_at"
]


## 4. Fonctions utilitaires : identifiants, logs, I/O et normalisation texte

Ces fonctions sont réutilisées par toutes les couches :
- création d'un `run_id` ;
- identifiants stables à partir du hash documentaire ;
- logs structurés ;
- écriture Parquet avec fallback CSV si `pyarrow` n'est pas disponible ;
- nettoyage de texte et JSON safe.


In [ ]:
# ============================================================
# 4. FONCTIONS UTILITAIRES
# ============================================================

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def make_run_id(prefix: str = "phase2") -> str:
    return f"{prefix}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"


def stable_id(*parts: Any, prefix: Optional[str] = None) -> str:
    raw = "|".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:18]
    return f"{prefix}_{digest}" if prefix else digest


def compute_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).replace(" ", " ").replace("﻿", "")
    text = re.sub(r"[ 	]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compact_text(text: Any) -> str:
    return re.sub(r"\s+", " ", normalize_text(text)).strip()


def safe_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, default=str)


def load_json(value: Any, default: Any = None) -> Any:
    if default is None:
        default = []
    if value is None or value == "":
        return default
    if isinstance(value, (list, dict)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return default


def ensure_columns(df, columns: List[str]):
    for col in columns:
        if col not in df.columns:
            df[col] = None
    return df[columns]


def write_table(df, path: Path, columns: List[str]):
    """Écrit en Parquet si possible, sinon en CSV fallback."""
    if pd is None:
        raise RuntimeError("pandas est requis pour écrire les tables de sortie")
    path.parent.mkdir(parents=True, exist_ok=True)
    df = ensure_columns(df.copy(), columns)
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".csv")
        df.to_csv(fallback, index=False)
        print(f"[WARN] Écriture Parquet impossible ({type(exc).__name__}). Fallback CSV : {fallback}")
        return fallback


def append_log(run_id: str, stage: str, document_row: Optional[Dict[str, Any]] = None,
               status: str = "OK", error_type: Optional[str] = None, error_message: Optional[str] = None):
    if pd is None:
        print("[LOG]", stage, status, error_type, error_message)
        return
    document_row = document_row or {}
    row = {
        "run_id": run_id,
        "stage": stage,
        "document_id": document_row.get("document_id"),
        "company": document_row.get("company"),
        "fiscal_year": document_row.get("fiscal_year"),
        "doc_subtype": document_row.get("doc_subtype"),
        "source_pdf_path": document_row.get("local_path") or document_row.get("source_pdf_path"),
        "status": status,
        "error_type": error_type,
        "error_message": error_message,
        "created_at": now_utc(),
    }
    frame = pd.DataFrame([row], columns=EXTRACTION_LOG_COLUMNS)
    frame.to_csv(EXTRACTION_LOG_PATH, mode="a" if EXTRACTION_LOG_PATH.exists() else "w", header=not EXTRACTION_LOG_PATH.exists(), index=False)


## 5. Chargement des documents de la phase 1

Cette cellule est le point d'ancrage avec la phase 1.

Elle lit `documents_metadata.csv`, conserve les PDFs réellement présents localement, récupère les métadonnées d'entreprise et de type documentaire, puis crée un `document_id` stable. Elle peut aussi filtrer sur les documents effectivement sélectionnés en phase 1 si `selected_documents.csv` est disponible.


In [ ]:
# ============================================================
# 5. CHARGEMENT DES DOCUMENTS INGÉRÉS EN PHASE 1
# ============================================================

def require_pandas():
    if pd is None:
        raise RuntimeError("pandas est requis. Installer les dépendances avec `pip install -r requirements-phase2.txt`.")


def load_selected_urls(selection_path: Path = SELECTED_DOCUMENTS_PATH) -> set:
    require_pandas()
    if not selection_path.exists():
        return set()
    selection = pd.read_csv(selection_path)
    if "selection_status" in selection.columns:
        selection = selection[selection["selection_status"].eq("SELECTED_FOR_DOWNLOAD")].copy()
    if "selected_url" not in selection.columns:
        return set()
    return set(selection["selected_url"].dropna().astype(str))


def load_phase1_documents(metadata_path: Path = DOCUMENT_METADATA_PATH,
                          selected_only: bool = False) -> "pd.DataFrame":
    require_pandas()
    if not metadata_path.exists():
        print(f"[WARN] Fichier de métadonnées introuvable : {metadata_path}")
        return pd.DataFrame()

    docs = pd.read_csv(metadata_path)
    if "local_path" not in docs.columns:
        raise ValueError("documents_metadata.csv doit contenir une colonne `local_path`")

    docs = docs[docs["local_path"].notna()].copy()
    docs["local_path"] = docs["local_path"].astype(str)
    docs["file_exists"] = docs["local_path"].map(lambda p: Path(p).exists())
    docs = docs[docs["file_exists"]].copy()

    if selected_only and "source_url" in docs.columns:
        selected_urls = load_selected_urls()
        if selected_urls:
            docs = docs[docs["source_url"].astype(str).isin(selected_urls)].copy()

    for col in ["company", "ticker", "isin", "jurisdiction", "fiscal_year", "doc_subtype", "source_url", "sha256"]:
        if col not in docs.columns:
            docs[col] = None

    missing_hash = docs["sha256"].isna() | docs["sha256"].astype(str).eq("")
    docs.loc[missing_hash, "sha256"] = docs.loc[missing_hash, "local_path"].map(lambda p: compute_sha256(Path(p)))
    docs["document_id"] = docs.apply(lambda r: stable_id(r.get("sha256") or r.get("local_path"), prefix="doc"), axis=1)

    docs = docs.drop_duplicates("document_id").reset_index(drop=True)
    print(f"Documents phase 1 prêts pour extraction : {len(docs)}")
    return docs


# Exemple d'utilisation après la phase 1 :
# phase1_documents = load_phase1_documents(selected_only=False)
# phase1_documents.head()


## 6. Registre d'indicateurs ESG

La collecte de la phase 1 était pilotée par un registre documentaire. La phase 2 suit la même philosophie : l'extraction est pilotée par un registre d'indicateurs.

Le registre versionné `data/esg_metric_registry_catalog.csv` définit les métriques, alias multilingues, unités attendues, types de valeur et bornes de validation.


In [ ]:
# ============================================================
# 6. REGISTRE D'INDICATEURS ESG
# ============================================================

DEFAULT_METRIC_REGISTRY = [
    {"metric_name":"scope_1_emissions","category":"climate","aliases":["scope 1","scope 1 emissions","scope 1 ghg","direct ghg emissions","émissions scope 1","émissions directes"],"expected_units":["tco2e","ktco2e","mtco2e","tonnes co2e","tons co2e"],"value_type":"numeric","min_value":0,"max_value":None,"priority":1},
    {"metric_name":"scope_2_emissions_location_based","category":"climate","aliases":["scope 2 location-based","scope 2 location based","location-based scope 2"],"expected_units":["tco2e","ktco2e","mtco2e"],"value_type":"numeric","min_value":0,"max_value":None,"priority":1},
    {"metric_name":"scope_2_emissions_market_based","category":"climate","aliases":["scope 2 market-based","scope 2 market based","market-based scope 2"],"expected_units":["tco2e","ktco2e","mtco2e"],"value_type":"numeric","min_value":0,"max_value":None,"priority":1},
    {"metric_name":"scope_3_emissions","category":"climate","aliases":["scope 3","scope 3 emissions","scope 3 ghg","indirect emissions","émissions scope 3"],"expected_units":["tco2e","ktco2e","mtco2e"],"value_type":"numeric","min_value":0,"max_value":None,"priority":1},
    {"metric_name":"total_ghg_emissions","category":"climate","aliases":["total ghg emissions","total greenhouse gas emissions","total emissions","émissions totales de ges"],"expected_units":["tco2e","ktco2e","mtco2e"],"value_type":"numeric","min_value":0,"max_value":None,"priority":1},
    {"metric_name":"energy_consumption","category":"energy","aliases":["energy consumption","total energy consumption","consommation d'énergie","consommation énergétique"],"expected_units":["kwh","mwh","gwh","tj"],"value_type":"numeric","min_value":0,"max_value":None,"priority":1},
    {"metric_name":"renewable_energy_share","category":"energy","aliases":["renewable energy share","renewable electricity","part d'énergie renouvelable","énergies renouvelables"],"expected_units":["%","percent","percentage"],"value_type":"percentage","min_value":0,"max_value":100,"priority":2},
    {"metric_name":"net_zero_target_year","category":"climate_target","aliases":["net zero by","net-zero by","carbon neutrality by","neutralité carbone","zéro émission nette"],"expected_units":["year"],"value_type":"year","min_value":2025,"max_value":2100,"priority":1},
    {"metric_name":"women_on_board","category":"governance","aliases":["women on the board","women on board","female board members","femmes au conseil"],"expected_units":["%","percent","percentage"],"value_type":"percentage","min_value":0,"max_value":100,"priority":2},
    {"metric_name":"board_independence","category":"governance","aliases":["board independence","independent directors","administrateurs indépendants","indépendance du conseil"],"expected_units":["%","percent","percentage"],"value_type":"percentage","min_value":0,"max_value":100,"priority":2},
    {"metric_name":"employees","category":"social","aliases":["number of employees","employees","headcount","effectif","nombre de salariés"],"expected_units":["count","fte"],"value_type":"integer","min_value":0,"max_value":None,"priority":2},
    {"metric_name":"ltifr","category":"social","aliases":["lost time injury frequency rate","ltifr","taux de fréquence","accidents avec arrêt"],"expected_units":["rate"],"value_type":"numeric","min_value":0,"max_value":None,"priority":2},
]


def split_pipe(value: Any) -> List[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    return [part.strip() for part in str(value).split("|") if part.strip()]


def load_metric_registry(path: Path = METRIC_REGISTRY_PATH) -> List[Dict[str, Any]]:
    if pd is None or not path.exists():
        print("[INFO] Registre CSV indisponible : utilisation du registre par défaut embarqué dans le notebook")
        return DEFAULT_METRIC_REGISTRY

    registry_df = pd.read_csv(path)
    registry = []
    for _, row in registry_df.iterrows():
        registry.append({
            "metric_name": row["metric_name"],
            "category": row["category"],
            "aliases": split_pipe(row.get("aliases")),
            "expected_units": split_pipe(row.get("expected_units")),
            "value_type": row.get("value_type"),
            "min_value": None if pd.isna(row.get("min_value")) else float(row.get("min_value")),
            "max_value": None if pd.isna(row.get("max_value")) else float(row.get("max_value")),
            "priority": int(row.get("priority", 2)),
        })
    print(f"Métriques chargées : {len(registry)}")
    return registry


ESG_METRIC_REGISTRY = load_metric_registry()
METRIC_BY_NAME = {spec["metric_name"]: spec for spec in ESG_METRIC_REGISTRY}


## 7. Routing ESG des pages et blocs

Avant d'extraire des métriques, le pipeline identifie les pages pertinentes. Cette étape réduit le bruit, accélère l'extraction et permet de documenter pourquoi une page a été considérée comme ESG.

La méthode reste explicable : dictionnaires de mots-clés par thème, score par section, labels multi-thématiques.


In [ ]:
# ============================================================
# 7. ROUTING ESG DES PAGES ET BLOCS
# ============================================================

ESG_SECTION_KEYWORDS = {
    "climate": ["climate", "ghg", "greenhouse", "scope 1", "scope 2", "scope 3", "emissions", "co2", "carbon", "carbone", "climat", "ges"],
    "energy": ["energy", "electricity", "renewable", "mwh", "gwh", "énergie", "électricité", "renouvelable"],
    "water": ["water", "withdrawal", "consumption", "eau", "prélèvement"],
    "waste": ["waste", "recycling", "circular", "déchets", "recyclage"],
    "social": ["employees", "workforce", "diversity", "safety", "training", "ltifr", "salariés", "sécurité", "formation"],
    "governance": ["board", "directors", "independence", "ethics", "corruption", "conseil", "gouvernance"],
    "assurance": ["assurance", "limited assurance", "reasonable assurance", "auditor", "vérification", "commissaire"],
    "taxonomy": ["taxonomy", "eligible", "aligned", "taxonomie", "éligible", "aligné"],
    "targets": ["target", "objective", "net zero", "sbti", "transition plan", "objectif", "neutralité carbone"],
}


def classify_esg_sections(text: Any, min_score: int = 1) -> Tuple[List[str], Dict[str, int]]:
    text_l = compact_text(text).lower()
    labels, scores = [], {}
    for label, keywords in ESG_SECTION_KEYWORDS.items():
        score = sum(1 for keyword in keywords if keyword.lower() in text_l)
        if score >= min_score:
            labels.append(label)
            scores[label] = score
    return labels, scores


def estimate_language(text: Any) -> str:
    text_l = f" {compact_text(text).lower()} "
    fr_hits = sum(1 for word in [" le ", " la ", " les ", " émissions ", " conseil ", " salariés ", " durabilité "] if word in text_l)
    en_hits = sum(1 for word in [" the ", " and ", " emissions ", " board ", " employees ", " sustainability "] if word in text_l)
    if fr_hits > en_hits:
        return "fr"
    if en_hits > fr_hits:
        return "en"
    return "unknown"


def extraction_quality_score(text: Any, has_tables: bool, has_images: bool) -> float:
    length = len(compact_text(text))
    score = 0.0
    if length >= MIN_TEXT_CHARS_FOR_PAGE:
        score += 0.40
    if length >= 500:
        score += 0.25
    if has_tables:
        score += 0.25
    if has_images:
        score += 0.10
    return round(min(score, 1.0), 3)


## 8. Parsing multimodal des PDFs : texte, blocs, tables et images

Cette cellule est le cœur de la phase 2.

Pour chaque PDF de la phase 1, elle extrait :
- texte page par page ;
- blocs de texte avec coordonnées ;
- tables avec `pdfplumber` ;
- images avec PyMuPDF ;
- OCR optionnel sur images si l'environnement le permet.

Le parsing est tolérant aux erreurs : un PDF ou une table problématique est loggé sans bloquer tout le run.


In [ ]:
# ============================================================
# 8. PARSING MULTIMODAL DES PDFS
# ============================================================

def require_pdf_dependencies():
    missing = [name for name, module in REQUIRED_FOR_FULL_RUN.items() if module is None]
    if missing:
        raise RuntimeError(f"Dépendances manquantes pour parser les PDFs : {missing}. Installer `requirements-phase2.txt`.")


def extract_text_blocks(run_id: str, page: Any, doc: Dict[str, Any], page_number: int) -> List[Dict[str, Any]]:
    rows = []
    for block_order, block in enumerate(page.get_text("blocks") or []):
        x0, y0, x1, y1, text, *_ = block
        text = normalize_text(text)
        if not text:
            continue
        labels, _ = classify_esg_sections(text)
        block_id = stable_id(doc["document_id"], page_number, block_order, text[:100], prefix="blk")
        rows.append({
            "run_id": run_id, "block_id": block_id, "document_id": doc["document_id"],
            "company": doc.get("company"), "fiscal_year": doc.get("fiscal_year"), "doc_subtype": doc.get("doc_subtype"),
            "source_pdf_path": doc.get("local_path"), "page_number": page_number, "block_type": "text",
            "block_order": block_order, "content": text,
            "bbox": safe_json([round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)]),
            "section_labels": safe_json(labels), "confidence": 0.90, "created_at": now_utc(),
        })
    return rows


def extract_images(run_id: str, pdf_doc: Any, page: Any, doc: Dict[str, Any], page_number: int) -> List[Dict[str, Any]]:
    rows = []
    for image_order, image in enumerate(page.get_images(full=True) or []):
        xref = image[0]
        image_id = stable_id(doc["document_id"], page_number, image_order, xref, prefix="img")
        image_path = IMAGE_DIR / f"{image_id}.png"
        width, height, ocr_text = None, None, None
        ocr_status = "OCR_DISABLED"
        try:
            pix = fitz.Pixmap(pdf_doc, xref)
            if pix.alpha:
                pix = fitz.Pixmap(fitz.csRGB, pix)
            width, height = pix.width, pix.height
            pix.save(str(image_path))
            if ENABLE_OCR and OCR_ENGINE_AVAILABLE:
                img = PIL_Image.open(image_path)
                ocr_text = normalize_text(pytesseract.image_to_string(img))
                ocr_status = "OCR_DONE" if ocr_text else "OCR_EMPTY"
        except Exception as exc:
            ocr_status = "IMAGE_EXTRACTION_ERROR"
            ocr_text = f"{type(exc).__name__}: {exc}"
        rows.append({
            "run_id": run_id, "image_id": image_id, "document_id": doc["document_id"],
            "company": doc.get("company"), "fiscal_year": doc.get("fiscal_year"), "doc_subtype": doc.get("doc_subtype"),
            "source_pdf_path": doc.get("local_path"), "page_number": page_number, "image_order": image_order,
            "image_path": str(image_path) if image_path.exists() else None, "width": width, "height": height,
            "ocr_text": ocr_text, "ocr_status": ocr_status, "created_at": now_utc(),
        })
    return rows


def extract_tables(run_id: str, doc: Dict[str, Any], max_pages: Optional[int] = MAX_PAGES_PER_DOCUMENT) -> List[Dict[str, Any]]:
    rows = []
    try:
        with pdfplumber.open(doc["local_path"]) as pdf:
            pages = pdf.pages if max_pages is None else pdf.pages[:max_pages]
            for page_index, page in enumerate(pages, start=1):
                for table_order, table in enumerate(page.extract_tables() or []):
                    cleaned = []
                    for row in table:
                        clean_row = [normalize_text(cell) if cell is not None else "" for cell in row]
                        if any(clean_row):
                            cleaned.append(clean_row)
                    if not cleaned:
                        continue
                    table_id = stable_id(doc["document_id"], page_index, table_order, safe_json(cleaned[:3]), prefix="tbl")
                    rows.append({
                        "run_id": run_id, "table_id": table_id, "document_id": doc["document_id"],
                        "company": doc.get("company"), "fiscal_year": doc.get("fiscal_year"), "doc_subtype": doc.get("doc_subtype"),
                        "source_pdf_path": doc.get("local_path"), "page_number": page_index, "table_order": table_order,
                        "n_rows": len(cleaned), "n_cols": max(len(r) for r in cleaned), "table_json": safe_json(cleaned),
                        "extraction_method": "pdfplumber", "created_at": now_utc(),
                    })
    except Exception as exc:
        append_log(run_id, "table_extraction", doc, status="ERROR", error_type=type(exc).__name__, error_message=str(exc))
    return rows


def parse_pdf_document(run_id: str, doc: Dict[str, Any], max_pages: Optional[int] = MAX_PAGES_PER_DOCUMENT):
    require_pdf_dependencies()
    page_rows, block_rows, image_rows = [], [], []
    try:
        pdf_doc = fitz.open(doc["local_path"])
        page_limit = len(pdf_doc) if max_pages is None else min(len(pdf_doc), max_pages)
        for page_index in range(page_limit):
            page_number = page_index + 1
            page = pdf_doc[page_index]
            text = normalize_text(page.get_text("text"))
            blocks = extract_text_blocks(run_id, page, doc, page_number)
            images = extract_images(run_id, pdf_doc, page, doc, page_number)
            labels, scores = classify_esg_sections(text)
            page_rows.append({
                "run_id": run_id, "document_id": doc["document_id"], "company": doc.get("company"),
                "ticker": doc.get("ticker"), "isin": doc.get("isin"), "jurisdiction": doc.get("jurisdiction"),
                "fiscal_year": doc.get("fiscal_year"), "doc_subtype": doc.get("doc_subtype"),
                "source_pdf_path": doc.get("local_path"), "sha256": doc.get("sha256"), "page_number": page_number,
                "text": text, "text_length": len(compact_text(text)), "has_tables": False, "has_images": len(images) > 0,
                "language": estimate_language(text), "section_labels": safe_json(labels), "section_scores": safe_json(scores),
                "extraction_quality_score": extraction_quality_score(text, False, len(images) > 0), "created_at": now_utc(),
            })
            block_rows.extend(blocks)
            image_rows.extend(images)
        pdf_doc.close()
        table_rows = extract_tables(run_id, doc, max_pages=max_pages)
        table_pages = {row["page_number"] for row in table_rows}
        for page_row in page_rows:
            page_row["has_tables"] = page_row["page_number"] in table_pages
            page_row["extraction_quality_score"] = extraction_quality_score(page_row["text"], page_row["has_tables"], page_row["has_images"])
        append_log(run_id, "parse_pdf_document", doc, status="OK")
        return page_rows, block_rows, table_rows, image_rows
    except Exception as exc:
        append_log(run_id, "parse_pdf_document", doc, status="ERROR", error_type=type(exc).__name__, error_message=str(exc))
        return page_rows, block_rows, [], image_rows


def run_pdf_parsing(run_id: str, documents_df) -> Tuple[Any, Any, Any, Any]:
    require_pandas()
    all_pages, all_blocks, all_tables, all_images = [], [], [], []
    for _, row in documents_df.iterrows():
        pages, blocks, tables, images = parse_pdf_document(run_id, row.to_dict())
        all_pages.extend(pages)
        all_blocks.extend(blocks)
        all_tables.extend(tables)
        all_images.extend(images)
    pages_df = pd.DataFrame(all_pages, columns=PARSED_PAGE_COLUMNS)
    blocks_df = pd.DataFrame(all_blocks, columns=DOCUMENT_BLOCK_COLUMNS)
    tables_df = pd.DataFrame(all_tables, columns=EXTRACTED_TABLE_COLUMNS)
    images_df = pd.DataFrame(all_images, columns=EXTRACTED_IMAGE_COLUMNS)
    write_table(pages_df, PARSED_PAGES_PATH, PARSED_PAGE_COLUMNS)
    write_table(blocks_df, DOCUMENT_BLOCKS_PATH, DOCUMENT_BLOCK_COLUMNS)
    write_table(tables_df, EXTRACTED_TABLES_PATH, EXTRACTED_TABLE_COLUMNS)
    write_table(images_df, EXTRACTED_IMAGES_PATH, EXTRACTED_IMAGE_COLUMNS)
    return pages_df, blocks_df, tables_df, images_df


## 9. Normalisation des valeurs et unités

Cette cellule standardise les valeurs extraites avant validation.

Elle gère notamment : séparateurs français/anglais, unités carbone `tCO2e`, `ktCO2e`, `MtCO2e`, énergie `kWh/MWh/GWh/TJ`, pourcentages, années cibles et valeurs entières.


In [ ]:
# ============================================================
# 9. NORMALISATION DES VALEURS ET UNITÉS
# ============================================================

UNIT_ALIASES = {
    "tco2e":"tCO2e", "t co2e":"tCO2e", "tonnes co2e":"tCO2e", "tons co2e":"tCO2e",
    "ktco2e":"ktCO2e", "kt co2e":"ktCO2e", "ktonnes co2e":"ktCO2e",
    "mtco2e":"MtCO2e", "mt co2e":"MtCO2e", "million tonnes co2e":"MtCO2e",
    "kwh":"kWh", "mwh":"MWh", "gwh":"GWh", "tj":"TJ",
    "%":"%", "percent":"%", "percentage":"%", "year":"year", "count":"count", "fte":"FTE", "rate":"rate",
}
UNIT_TO_BASE = {
    "tCO2e":("tCO2e", 1.0), "ktCO2e":("tCO2e", 1_000.0), "MtCO2e":("tCO2e", 1_000_000.0),
    "kWh":("MWh", 0.001), "MWh":("MWh", 1.0), "GWh":("MWh", 1_000.0), "TJ":("MWh", 277.777778),
    "%":("%", 1.0), "year":("year", 1.0), "count":("count", 1.0), "FTE":("FTE", 1.0), "rate":("rate", 1.0),
}
NUMBER_PATTERN = r"[-+]?\d{1,3}(?:[\s,\.]\d{3})*(?:[\.,]\d+)?|[-+]?\d+(?:[\.,]\d+)?"
UNIT_PATTERN = r"(MtCO2e|ktCO2e|tCO2e|Mt CO2e|kt CO2e|t CO2e|tonnes CO2e|tons CO2e|GWh|MWh|kWh|TJ|%|percent|percentage|FTE|count|rate)"


def parse_number(value: Any) -> Optional[float]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip().replace(" ", " ")
    text = re.sub(r"[^0-9,\.\-+ ]", "", text).strip()
    if not text:
        return None
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".") if text.rfind(",") > text.rfind(".") else text.replace(",", "")
    elif "," in text:
        parts = text.split(",")
        text = text.replace(" ", "").replace(",", ".") if len(parts[-1]) in [1, 2] else text.replace(",", "").replace(" ", "")
    else:
        text = text.replace(" ", "")
        if text.count(".") > 1:
            text = text.replace(".", "")
    try:
        return float(text)
    except ValueError:
        return None


def normalize_unit(unit: Any) -> Optional[str]:
    if unit is None or unit == "":
        return None
    key = compact_text(unit).lower().replace("₂", "2").replace("co₂", "co2")
    return UNIT_ALIASES.get(key, str(unit))


def normalize_value_unit(raw_value: Any, raw_unit: Any) -> Tuple[Optional[float], Optional[str]]:
    value = parse_number(raw_value)
    unit = normalize_unit(raw_unit)
    if value is None:
        return None, unit
    if unit in UNIT_TO_BASE:
        base_unit, factor = UNIT_TO_BASE[unit]
        return value * factor, base_unit
    return value, unit


def extract_years(text: Any) -> List[int]:
    return [int(y) for y in re.findall(r"(19\d{2}|20\d{2})", str(text))]


## 10. Extraction des métriques depuis les tables

Les tableaux sont privilégiés car ils contiennent souvent les métriques ESG les plus structurées.

La fonction parcourt les tables, détecte les lignes correspondant au registre d'indicateurs, identifie les colonnes d'années, extrait la valeur correspondant à l'exercice fiscal et génère un candidat métrique avec preuve source.


In [ ]:
# ============================================================
# 10. EXTRACTION DEPUIS LES TABLES
# ============================================================

def find_metric_specs(text: Any) -> List[Dict[str, Any]]:
    text_l = compact_text(text).lower()
    matches = []
    for spec in ESG_METRIC_REGISTRY:
        if any(alias.lower() in text_l for alias in spec.get("aliases", [])):
            matches.append(spec)
    return matches


def detect_unit(text: Any, spec: Optional[Dict[str, Any]] = None) -> Optional[str]:
    match = re.search(UNIT_PATTERN, str(text), flags=re.IGNORECASE)
    if match:
        return normalize_unit(match.group(1))
    if spec:
        if spec.get("value_type") == "percentage":
            return "%"
        if spec.get("value_type") == "year":
            return "year"
        if spec.get("value_type") == "integer":
            return "count"
    return None


def detect_year_columns(table: List[List[str]]) -> Dict[int, int]:
    year_cols = {}
    for header in table[:min(3, len(table))]:
        for col_idx, cell in enumerate(header):
            for year in extract_years(cell):
                year_cols[year] = col_idx
    return year_cols


def row_to_text(row: Iterable[Any]) -> str:
    return compact_text(" | ".join(str(cell) for cell in row if cell is not None))


def choose_value_from_row(row: List[str], fiscal_year: Optional[int], year_cols: Dict[int, int], row_text_value: str) -> Optional[str]:
    if fiscal_year in year_cols and year_cols[fiscal_year] < len(row):
        return row[year_cols[fiscal_year]]
    numbers = re.findall(NUMBER_PATTERN, row_text_value)
    parsed = [(num, parse_number(num)) for num in numbers]
    parsed = [(num, val) for num, val in parsed if val is not None and not (1900 <= val <= 2100 and float(val).is_integer())]
    return parsed[-1][0] if parsed else None


def extract_metric_candidates_from_tables(run_id: str, tables_df, documents_df):
    require_pandas()
    if tables_df.empty:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
    doc_lookup = documents_df.set_index("document_id").to_dict("index") if not documents_df.empty else {}
    candidates = []
    for _, table_row in tables_df.iterrows():
        table = load_json(table_row.get("table_json"), default=[])
        fiscal_year = int(table_row["fiscal_year"]) if str(table_row.get("fiscal_year", "")).replace(".0", "").isdigit() else None
        year_cols = detect_year_columns(table)
        doc_meta = doc_lookup.get(table_row["document_id"], {})
        for row_idx, row in enumerate(table):
            text = row_to_text(row)
            for spec in find_metric_specs(text):
                raw_value = choose_value_from_row(row, fiscal_year, year_cols, text)
                if raw_value is None:
                    continue
                raw_unit = detect_unit(text, spec)
                normalized_value, normalized_unit = normalize_value_unit(raw_value, raw_unit)
                candidates.append({
                    "run_id": run_id,
                    "metric_id": stable_id(table_row["table_id"], row_idx, spec["metric_name"], raw_value, prefix="met"),
                    "document_id": table_row["document_id"], "company": table_row.get("company"),
                    "ticker": doc_meta.get("ticker"), "isin": doc_meta.get("isin"), "jurisdiction": doc_meta.get("jurisdiction"),
                    "fiscal_year": fiscal_year, "doc_subtype": table_row.get("doc_subtype"),
                    "metric_name": spec["metric_name"], "metric_category": spec["category"],
                    "raw_value": raw_value, "raw_unit": raw_unit, "normalized_value": normalized_value, "normalized_unit": normalized_unit,
                    "source_page": table_row.get("page_number"), "source_block_id": table_row.get("table_id"),
                    "source_text_excerpt": text[:800], "extraction_method": "table_parser", "confidence_score": 0.88,
                    "validation_status": "pending_validation", "quality_flags": safe_json(["from_table"]), "created_at": now_utc(),
                })
    return pd.DataFrame(candidates, columns=METRIC_CANDIDATE_COLUMNS)


## 11. Extraction depuis texte et OCR

Certaines informations importantes ne sont pas dans des tableaux : objectif net zero, année cible, existence d'une validation, politique de transition, éléments de gouvernance.

Cette cellule extrait des candidats à partir des blocs texte et des textes OCR en cherchant les alias du registre puis les valeurs proches dans le contexte.


In [ ]:
# ============================================================
# 11. EXTRACTION DEPUIS TEXTE ET OCR
# ============================================================

def context_window(text: str, start: int, end: int, window: int = CONTEXT_WINDOW_CHARS) -> str:
    return compact_text(text[max(0, start - window): min(len(text), end + window)])


def extract_value_from_context(context: str, spec: Dict[str, Any]) -> Tuple[Optional[Any], Optional[str]]:
    if spec.get("value_type") == "year":
        years = [year for year in extract_years(context) if 2025 <= year <= 2100]
        return (years[0], "year") if years else (None, "year")
    unit = detect_unit(context, spec)
    parsed = [(num, parse_number(num)) for num in re.findall(NUMBER_PATTERN, context)]
    parsed = [(num, val) for num, val in parsed if val is not None]
    parsed = [(num, val) for num, val in parsed if not (1900 <= val <= 2100 and float(val).is_integer())]
    if spec.get("value_type") == "percentage":
        parsed = [(num, val) for num, val in parsed if 0 <= val <= 100]
        unit = "%"
    if not parsed:
        return None, unit
    return parsed[0][0], unit


def extract_metric_candidates_from_blocks(run_id: str, blocks_df, documents_df, method: str = "text_block"):
    require_pandas()
    if blocks_df.empty:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
    doc_lookup = documents_df.set_index("document_id").to_dict("index") if not documents_df.empty else {}
    candidates = []
    for _, block in blocks_df.iterrows():
        text = compact_text(block.get("content"))
        if len(text) < 20:
            continue
        doc_meta = doc_lookup.get(block["document_id"], {})
        for spec in ESG_METRIC_REGISTRY:
            for alias in spec.get("aliases", []):
                match = re.search(re.escape(alias), text, flags=re.IGNORECASE)
                if not match:
                    continue
                context = context_window(text, match.start(), match.end())
                raw_value, raw_unit = extract_value_from_context(context, spec)
                if raw_value is None:
                    continue
                normalized_value, normalized_unit = normalize_value_unit(raw_value, raw_unit)
                candidates.append({
                    "run_id": run_id,
                    "metric_id": stable_id(block["block_id"], spec["metric_name"], raw_value, prefix="met"),
                    "document_id": block["document_id"], "company": block.get("company"),
                    "ticker": doc_meta.get("ticker"), "isin": doc_meta.get("isin"), "jurisdiction": doc_meta.get("jurisdiction"),
                    "fiscal_year": block.get("fiscal_year") or doc_meta.get("fiscal_year"), "doc_subtype": block.get("doc_subtype"),
                    "metric_name": spec["metric_name"], "metric_category": spec["category"],
                    "raw_value": raw_value, "raw_unit": raw_unit, "normalized_value": normalized_value, "normalized_unit": normalized_unit,
                    "source_page": block.get("page_number"), "source_block_id": block.get("block_id"),
                    "source_text_excerpt": context, "extraction_method": method,
                    "confidence_score": 0.70 if method == "text_block" else 0.55,
                    "validation_status": "pending_validation", "quality_flags": safe_json([f"from_{method}"]), "created_at": now_utc(),
                })
                break
    return pd.DataFrame(candidates, columns=METRIC_CANDIDATE_COLUMNS)


def build_ocr_blocks(run_id: str, images_df):
    require_pandas()
    if images_df.empty or "ocr_text" not in images_df.columns:
        return pd.DataFrame(columns=DOCUMENT_BLOCK_COLUMNS)
    rows = []
    ocr_images = images_df[images_df["ocr_text"].notna() & images_df["ocr_text"].astype(str).str.len().gt(10)]
    for _, image in ocr_images.iterrows():
        text = normalize_text(image.get("ocr_text"))
        labels, _ = classify_esg_sections(text)
        rows.append({
            "run_id": run_id, "block_id": stable_id(image["image_id"], "ocr", text[:100], prefix="blk"),
            "document_id": image["document_id"], "company": image.get("company"), "fiscal_year": image.get("fiscal_year"),
            "doc_subtype": image.get("doc_subtype"), "source_pdf_path": image.get("source_pdf_path"),
            "page_number": image.get("page_number"), "block_type": "ocr_image", "block_order": image.get("image_order"),
            "content": text, "bbox": None, "section_labels": safe_json(labels), "confidence": 0.55, "created_at": now_utc(),
        })
    return pd.DataFrame(rows, columns=DOCUMENT_BLOCK_COLUMNS)


## 12. Validation métier, scoring et revue manuelle

Cette cellule applique les contrôles qualité qui rendent le pipeline défendable :
- valeur présente ;
- unité cohérente ;
- bornes économiques ;
- preuve source ;
- pénalisation OCR ou faible confiance ;
- conservation en revue manuelle plutôt que suppression brutale.


In [ ]:
# ============================================================
# 12. VALIDATION MÉTIER ET SCORE DE CONFIANCE
# ============================================================

def expected_base_units(spec: Dict[str, Any]) -> set:
    units = set()
    for unit in spec.get("expected_units", []):
        normalized = normalize_unit(unit)
        if normalized in UNIT_TO_BASE:
            units.add(UNIT_TO_BASE[normalized][0])
        elif normalized:
            units.add(normalized)
    return units


def validate_single_candidate(row: Dict[str, Any]) -> Tuple[str, float, List[str]]:
    spec = METRIC_BY_NAME.get(row.get("metric_name"))
    flags = load_json(row.get("quality_flags"), default=[])
    score = float(row.get("confidence_score") or 0.0)
    status = "validated"
    if spec is None:
        return "review_required", max(0.0, score - 0.20), sorted(set(flags + ["unknown_metric"]))

    value = row.get("normalized_value")
    unit = row.get("normalized_unit")
    if value is None or (isinstance(value, float) and math.isnan(value)):
        status = "rejected"
        score -= 0.35
        flags.append("missing_value")
    else:
        min_value, max_value = spec.get("min_value"), spec.get("max_value")
        if min_value is not None and value < min_value:
            status = "review_required"
            score -= 0.20
            flags.append("below_min")
        if max_value is not None and value > max_value:
            status = "review_required"
            score -= 0.20
            flags.append("above_max")

    bases = expected_base_units(spec)
    if unit and bases and unit not in bases:
        status = "review_required" if status != "rejected" else status
        score -= 0.15
        flags.append("unexpected_unit")

    if not row.get("source_page") or not row.get("source_text_excerpt"):
        status = "review_required" if status != "rejected" else status
        score -= 0.20
        flags.append("missing_source_evidence")

    if row.get("extraction_method") == "table_parser":
        score += 0.05
    if row.get("extraction_method") == "ocr_image":
        score -= 0.15
        flags.append("ocr_source")

    score = round(max(0.0, min(1.0, score)), 3)
    if score < 0.45 and status == "validated":
        status = "review_required"
        flags.append("low_confidence")
    return status, score, sorted(set(flags))


def validate_metric_candidates(candidates_df):
    require_pandas()
    if candidates_df.empty:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
    rows = []
    for _, row in candidates_df.iterrows():
        row_dict = row.to_dict()
        status, score, flags = validate_single_candidate(row_dict)
        row_dict["validation_status"] = status
        row_dict["confidence_score"] = score
        row_dict["quality_flags"] = safe_json(flags)
        rows.append(row_dict)
    return pd.DataFrame(rows, columns=METRIC_CANDIDATE_COLUMNS)


def build_review_queue(validated_metrics):
    require_pandas()
    review = validated_metrics[validated_metrics["validation_status"].isin(["review_required", "conflict_review_required"])].copy() if not validated_metrics.empty else pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
    REVIEW_QUEUE_PATH.parent.mkdir(parents=True, exist_ok=True)
    review.to_csv(REVIEW_QUEUE_PATH, index=False)
    return review


## 13. Construction du panel entreprise-année-indicateur

La sortie finale de la phase 2 est un panel analytique. Pour chaque couple entreprise-année-métrique, le pipeline conserve le meilleur candidat selon le statut et le score de confiance.

Ce panel est la table qui alimentera les étapes suivantes : scoring ESG, comparaison sectorielle, analyse de risque climat et signaux quantitatifs.


In [ ]:
# ============================================================
# 13. PANEL ENTREPRISE-ANNÉE-INDICATEUR
# ============================================================

def build_company_year_esg_panel(run_id: str, validated_metrics):
    require_pandas()
    if validated_metrics.empty:
        panel = pd.DataFrame(columns=COMPANY_YEAR_PANEL_COLUMNS)
        write_table(panel, COMPANY_YEAR_PANEL_PATH, COMPANY_YEAR_PANEL_COLUMNS)
        return panel
    usable = validated_metrics[validated_metrics["validation_status"].isin(["validated", "review_required"])].copy()
    usable = usable[usable["normalized_value"].notna()].copy()
    if usable.empty:
        panel = pd.DataFrame(columns=COMPANY_YEAR_PANEL_COLUMNS)
        write_table(panel, COMPANY_YEAR_PANEL_PATH, COMPANY_YEAR_PANEL_COLUMNS)
        return panel
    status_rank = {"validated": 0, "review_required": 1}
    usable["status_rank"] = usable["validation_status"].map(status_rank).fillna(9)
    usable = usable.sort_values(["company", "fiscal_year", "metric_name", "status_rank", "confidence_score"], ascending=[True, True, True, True, False])
    best = usable.groupby(["company", "fiscal_year", "metric_name"], dropna=False).head(1).copy()
    panel = pd.DataFrame({
        "run_id": run_id,
        "company": best["company"], "ticker": best["ticker"], "isin": best["isin"], "jurisdiction": best["jurisdiction"],
        "fiscal_year": best["fiscal_year"], "metric_name": best["metric_name"], "metric_category": best["metric_category"],
        "metric_value": best["normalized_value"], "metric_unit": best["normalized_unit"],
        "best_confidence_score": best["confidence_score"], "validation_status": best["validation_status"],
        "source_document_id": best["document_id"], "source_doc_subtype": best["doc_subtype"], "source_page": best["source_page"],
        "source_text_excerpt": best["source_text_excerpt"], "quality_flags": best["quality_flags"], "created_at": now_utc(),
    })
    panel = panel[COMPANY_YEAR_PANEL_COLUMNS].sort_values(["company", "fiscal_year", "metric_name"])
    write_table(panel, COMPANY_YEAR_PANEL_PATH, COMPANY_YEAR_PANEL_COLUMNS)
    return panel


## 14. Rapport qualité de la phase 2

Cette cellule produit un rapport synthétique : nombre de documents, pages, tables, images, métriques candidates, métriques validées, éléments en revue et ratio de couverture par rapport au registre d'indicateurs.

C'est utile en entretien pour montrer que le pipeline ne se contente pas d'extraire : il mesure sa propre qualité.


In [ ]:
# ============================================================
# 14. RAPPORT QUALITÉ
# ============================================================

def build_quality_report(run_id: str, documents_df, pages_df, tables_df, images_df, candidates_df, validated_df):
    require_pandas()
    n_documents = len(documents_df) if documents_df is not None else 0
    n_expected_metrics = max(1, n_documents * len(ESG_METRIC_REGISTRY))
    n_validated = int(validated_df["validation_status"].eq("validated").sum()) if not validated_df.empty else 0
    n_review = int(validated_df["validation_status"].eq("review_required").sum()) if not validated_df.empty else 0
    report = pd.DataFrame([{
        "run_id": run_id, "level": "run", "key": "global",
        "n_documents": n_documents, "n_pages": len(pages_df), "n_tables": len(tables_df), "n_images": len(images_df),
        "n_metric_candidates": len(candidates_df), "n_validated_metrics": n_validated, "n_review_required": n_review,
        "coverage_ratio": round(n_validated / n_expected_metrics, 4), "created_at": now_utc(),
    }], columns=QUALITY_REPORT_COLUMNS)
    report.to_csv(QUALITY_REPORT_PATH, index=False)
    return report


def summarize_phase2(panel, validated_metrics, quality_report):
    print("===== SYNTHÈSE PHASE 2 — EXTRACTION ESG =====")
    print("Panel final :", panel.shape)
    if not validated_metrics.empty:
        print("\nStatuts de validation :")
        print(validated_metrics["validation_status"].value_counts(dropna=False))
        print("\nMéthodes d'extraction :")
        print(validated_metrics["extraction_method"].value_counts(dropna=False))
    print("\nRapport qualité :")
    print(quality_report)


## 15. Orchestration complète de la phase 2

Cette cellule assemble l'ensemble de la chaîne : chargement phase 1, parsing, extraction depuis tables/textes/OCR, validation, revue, panel et rapport qualité.

Elle est conçue pour être appelée après l'exécution de la phase 1, lorsque `documents_metadata.csv` référence les PDFs téléchargés.


In [ ]:
# ============================================================
# 15. ORCHESTRATION COMPLÈTE
# ============================================================

def run_esg_phase2_extraction(documents_df=None, selected_only: bool = False, documents_limit: Optional[int] = None):
    require_pandas()
    run_id = make_run_id()
    if documents_df is None:
        documents_df = load_phase1_documents(selected_only=selected_only)
    if documents_limit is not None:
        documents_df = documents_df.head(documents_limit).copy()
    if documents_df.empty:
        print("Aucun document disponible. Exécuter la phase 1 ou vérifier `documents_metadata.csv`.")
        empty_candidates = pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
        empty_panel = pd.DataFrame(columns=COMPANY_YEAR_PANEL_COLUMNS)
        return {"run_id": run_id, "documents": documents_df, "metric_candidates": empty_candidates, "validated_metrics": empty_candidates, "panel": empty_panel}

    pages_df, blocks_df, tables_df, images_df = run_pdf_parsing(run_id, documents_df)
    table_candidates = extract_metric_candidates_from_tables(run_id, tables_df, documents_df)
    text_candidates = extract_metric_candidates_from_blocks(run_id, blocks_df, documents_df, method="text_block")
    ocr_blocks = build_ocr_blocks(run_id, images_df)
    ocr_candidates = extract_metric_candidates_from_blocks(run_id, ocr_blocks, documents_df, method="ocr_image")

    candidates_df = pd.concat([table_candidates, text_candidates, ocr_candidates], ignore_index=True)
    if not candidates_df.empty:
        candidates_df = candidates_df.drop_duplicates("metric_id")
    write_table(candidates_df, METRIC_CANDIDATES_PATH, METRIC_CANDIDATE_COLUMNS)

    validated_df = validate_metric_candidates(candidates_df)
    write_table(validated_df, VALIDATED_METRICS_PATH, METRIC_CANDIDATE_COLUMNS)
    review_df = build_review_queue(validated_df)
    panel_df = build_company_year_esg_panel(run_id, validated_df)
    quality_report = build_quality_report(run_id, documents_df, pages_df, tables_df, images_df, candidates_df, validated_df)
    summarize_phase2(panel_df, validated_df, quality_report)

    return {
        "run_id": run_id, "documents": documents_df, "pages": pages_df, "blocks": blocks_df,
        "tables": tables_df, "images": images_df, "ocr_blocks": ocr_blocks,
        "metric_candidates": candidates_df, "validated_metrics": validated_df,
        "review_queue": review_df, "panel": panel_df, "quality_report": quality_report,
    }


# Lancement complet après la phase 1 :
# results = run_esg_phase2_extraction(selected_only=False, documents_limit=None)


## 16. Mode démonstration sans PDF

Cette cellule permet de tester la logique métier même sans PDFs disponibles. Elle crée un document fictif, un bloc texte et une table ESG synthétique, puis exécute l'extraction, la validation et la construction du panel.

Ce mode est utile pour un Data Case : il rend la démonstration indépendante du réseau et des fichiers volumineux.


In [ ]:
# ============================================================
# 16. MODE DÉMONSTRATION SANS PDF
# ============================================================

def build_demo_phase2_inputs(run_id: str):
    require_pandas()
    demo_doc = pd.DataFrame([{
        "document_id": "doc_demo_2024", "company": "Demo Energy Corp", "ticker": "DEM", "isin": "DEMO00000001",
        "jurisdiction": "France", "fiscal_year": 2024, "doc_subtype": "sustainability_statement_csrd_esrs",
        "local_path": "demo.pdf", "sha256": "demo_sha256",
    }])
    demo_blocks = pd.DataFrame([{
        "run_id": run_id, "block_id": "blk_demo_targets", "document_id": "doc_demo_2024", "company": "Demo Energy Corp",
        "fiscal_year": 2024, "doc_subtype": "sustainability_statement_csrd_esrs", "source_pdf_path": "demo.pdf",
        "page_number": 42, "block_type": "text", "block_order": 1,
        "content": "The Group targets net zero by 2050 and reached 42% renewable electricity in 2024.",
        "bbox": None, "section_labels": safe_json(["climate", "energy", "targets"]), "confidence": 0.90, "created_at": now_utc(),
    }], columns=DOCUMENT_BLOCK_COLUMNS)
    table = [
        ["Indicator", "Unit", "2023", "2024"],
        ["Scope 1 GHG emissions", "ktCO2e", "120", "110"],
        ["Scope 2 market-based emissions", "ktCO2e", "50", "45"],
        ["Scope 3 emissions", "MtCO2e", "1.8", "1.6"],
        ["Women on board", "%", "43", "45"],
    ]
    demo_tables = pd.DataFrame([{
        "run_id": run_id, "table_id": "tbl_demo_esg", "document_id": "doc_demo_2024", "company": "Demo Energy Corp",
        "fiscal_year": 2024, "doc_subtype": "sustainability_statement_csrd_esrs", "source_pdf_path": "demo.pdf",
        "page_number": 41, "table_order": 1, "n_rows": len(table), "n_cols": 4,
        "table_json": safe_json(table), "extraction_method": "pdfplumber", "created_at": now_utc(),
    }], columns=EXTRACTED_TABLE_COLUMNS)
    return demo_doc, demo_blocks, demo_tables


def run_demo_phase2_extraction():
    require_pandas()
    run_id = make_run_id("demo_phase2")
    demo_doc, demo_blocks, demo_tables = build_demo_phase2_inputs(run_id)
    table_candidates = extract_metric_candidates_from_tables(run_id, demo_tables, demo_doc)
    text_candidates = extract_metric_candidates_from_blocks(run_id, demo_blocks, demo_doc, method="text_block")
    candidates_df = pd.concat([table_candidates, text_candidates], ignore_index=True).drop_duplicates("metric_id")
    validated_df = validate_metric_candidates(candidates_df)
    panel_df = build_company_year_esg_panel(run_id, validated_df)
    quality_report = build_quality_report(run_id, demo_doc, pd.DataFrame(), demo_tables, pd.DataFrame(), candidates_df, validated_df)
    summarize_phase2(panel_df, validated_df, quality_report)
    return {"run_id": run_id, "metric_candidates": candidates_df, "validated_metrics": validated_df, "panel": panel_df, "quality_report": quality_report}


# Démonstration :
# demo_results = run_demo_phase2_extraction()
# demo_results["panel"]


## 17. Extensions naturelles pour la suite du projet

Cette V1 est volontairement robuste et explicable. Les extensions prioritaires sont :

1. **Extraction LLM contrôlée** : appliquer un modèle uniquement sur des passages courts déjà routés, avec sortie JSON et interdiction d'inférer une valeur absente.
2. **Embeddings retrieval** : retrouver les passages les plus proches de chaque définition d'indicateur ESG.
3. **Validation cross-document** : comparer une métrique entre rapport annuel, déclaration de durabilité, rapport climat et rapport d'assurance.
4. **Interface de revue** : valider ou rejeter les lignes `review_required`.
5. **Scoring ESG** : transformer le panel entreprise-année-indicateur en signaux comparables par secteur.

Le point clé : la phase 2 reste dans la continuité de la phase 1, avec la même philosophie de modularité, traçabilité, scoring, statuts et auditabilité.
